# Online Retail II: EDA и прогноз дневной выручки

Датасет: транзакции британского интернет-магазина за 2009–2011 годы (UCI ML Repository).
Цель: показать полный цикл работы с «грязной» выгрузкой — профиль, очистка, EDA, признаки, модель, метрики и выводы.

Ноутбук читает полный файл `data/raw/online_retail_ii.csv`, если он есть, иначе выборку на 100 000 строк.


In [1]:
from pathlib import Path
import json

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data/raw"
PROCESSED = ROOT / "data/processed"
OUTPUTS = ROOT / "outputs"
FIGURES = OUTPUTS / "figures"
MODELS = ROOT / "models"
for folder in (PROCESSED, OUTPUTS, FIGURES, MODELS):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 20)
print("project root:", ROOT)

project root: D:\ML\Projects\kwork\eda_model


In [2]:
full_path = RAW / "online_retail_ii.csv"
sample_path = RAW / "online_retail_ii_sample.csv"
path = full_path if full_path.exists() else sample_path

df = pd.read_csv(path, parse_dates=["InvoiceDate"])
print("file:", path.name)
print("shape:", df.shape)
df.head()

file:

 online_retail_ii_sample.csv
shape: (100000, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year
0,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Year 2009-2010
1,489436,21754,HOME BUILDING BLOCK WORD,3,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,Year 2009-2010
2,489437,22143,CHRISTMAS CRAFT HEART DECORATIONS,6,2009-12-01 09:08:00,2.10,15362.0,United Kingdom,Year 2009-2010
3,489437,22130,PARTY CONE CHRISTMAS DECORATION,12,2009-12-01 09:08:00,0.85,15362.0,United Kingdom,Year 2009-2010
4,489437,21912,VINTAGE SNAKES & LADDERS,4,2009-12-01 09:08:00,3.75,15362.0,United Kingdom,Year 2009-2010


## 1. Профиль данных

Смотрим типы, пропуски, дубликаты и признаки «грязных» записей: отмены, отрицательные количества, нулевые цены.

In [3]:
df.info()

missing = df.isna().sum().to_frame("missing")
missing["share, %"] = (missing["missing"] / len(df) * 100).round(2)
missing[missing["missing"] > 0]

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      100000 non-null  str           
 1   StockCode    100000 non-null  str           
 2   Description  99610 non-null   str           
 3   Quantity     100000 non-null  int64         
 4   InvoiceDate  100000 non-null  datetime64[us]
 5   Price        100000 non-null  float64       
 6   Customer ID  77340 non-null   float64       
 7   Country      100000 non-null  str           
 8   Year         100000 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(5)
memory usage: 6.9 MB


,missing,"share, %"
Description,390,0.39
Customer ID,22660,22.66


In [4]:
duplicates = df.duplicated().sum()
cancelled = df["Invoice"].astype(str).str.startswith("C").sum()
negative_qty = (df["Quantity"] < 0).sum()
zero_price = (df["Price"] <= 0).sum()

print(f"duplicates: {duplicates}")
print(f"cancelled invoices: {cancelled} ({cancelled / len(df) * 100:.1f}%)")
print(f"negative quantity: {negative_qty} ({negative_qty / len(df) * 100:.1f}%)")
print(f"price <= 0: {zero_price} ({zero_price / len(df) * 100:.1f}%)")

duplicates: 105
cancelled invoices: 1879 (1.9%)
negative quantity: 2216 (2.2%)
price <= 0: 565 (0.6%)


## 2. Очистка

Убираем отмены и возвраты, нулевые цены, строки без клиента и дубликаты. Каждый шаг считается, чтобы было видно, сколько данных теряется.

In [5]:
raw_rows = len(df)
work = df.copy()

step1 = work[~work["Invoice"].astype(str).str.startswith("C")]
step2 = step1[step1["Quantity"] > 0]
step3 = step2[step2["Price"] > 0]
step4 = step3.dropna(subset=["Customer ID"])
cleaned = step4.drop_duplicates().copy()

cleaned["Revenue"] = cleaned["Quantity"] * cleaned["Price"]
cleaned["Hour"] = cleaned["InvoiceDate"].dt.hour
cleaned["DayOfWeek"] = cleaned["InvoiceDate"].dt.dayofweek
cleaned["Month"] = cleaned["InvoiceDate"].dt.month

print(f"raw:       {raw_rows:>8,}")
print(f"no cancel: {len(step1):>8,}")
print(f"qty > 0:   {len(step2):>8,}")
print(f"price > 0: {len(step3):>8,}")
print(f"customer:  {len(step4):>8,}")
print(f"dedup:     {len(cleaned):>8,} ({len(cleaned) / raw_rows * 100:.1f}% от исходных)")

raw:        100,000
no cancel:   98,121
qty > 0:     97,784
price > 0:   97,556
customer:    75,514
dedup:       75,412 (75.4% от исходных)


In [6]:
cleaned.to_csv(PROCESSED / "retail_cleaned.csv", index=False)
print("saved:", PROCESSED / "retail_cleaned.csv")
cleaned.head()

saved: D:\ML\Projects\kwork\eda_model\data\processed\retail_cleaned.csv


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year,Revenue,Hour,DayOfWeek,Month
0,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Year 2009-2010,59.50,7,1,12
1,489436,21754,HOME BUILDING BLOCK WORD,3,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,Year 2009-2010,17.85,9,1,12
2,489437,22143,CHRISTMAS CRAFT HEART DECORATIONS,6,2009-12-01 09:08:00,2.10,15362.0,United Kingdom,Year 2009-2010,12.60,9,1,12
3,489437,22130,PARTY CONE CHRISTMAS DECORATION,12,2009-12-01 09:08:00,0.85,15362.0,United Kingdom,Year 2009-2010,10.20,9,1,12
4,489437,21912,VINTAGE SNAKES & LADDERS,4,2009-12-01 09:08:00,3.75,15362.0,United Kingdom,Year 2009-2010,15.00,9,1,12


## 3. EDA

Выручка по месяцам, страны и товары-лидеры. Здесь видно сезонность и перекос по рынкам.

In [7]:
monthly = cleaned.set_index("InvoiceDate")["Revenue"].resample("ME").sum()

fig, ax = plt.subplots(figsize=(11, 4))
monthly.plot(ax=ax, marker="o", linewidth=1.5)
ax.set_title("Выручка по месяцам")
ax.set_xlabel("")
ax.set_ylabel("Выручка, £")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "revenue_by_month.png", dpi=150)
plt.show()

C:\Users\boroz\AppData\Local\Temp\ipykernel_12416\3794777171.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
top_countries = (
    cleaned.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(8)
)

fig, ax = plt.subplots(figsize=(8, 4))
top_countries.sort_values().plot.barh(ax=ax, color="#4C78A8")
ax.set_title("Выручка по странам, топ-8")
ax.set_xlabel("£")
fig.tight_layout()
fig.savefig(FIGURES / "top_countries.png", dpi=150)
plt.show()

top_countries.to_frame("revenue")

C:\Users\boroz\AppData\Local\Temp\ipykernel_12416\3898778233.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,revenue
Country,
United Kingdom,1333914.921
EIRE,56148.550
Netherlands,52032.570
Germany,38214.100
France,32616.350
Australia,17028.980
Spain,9459.570
Switzerland,9418.140


In [9]:
top_products = (
    cleaned.groupby("Description")["Revenue"].sum().sort_values(ascending=False).head(10)
)
top_products.to_frame("revenue")

,revenue
Description,
REGENCY CAKESTAND 3 TIER,25515.90
WHITE HANGING HEART T-LIGHT HOLDER,21611.60
ASSORTED COLOUR BIRD ORNAMENT,14967.13
JUMBO BAG RED RETROSPOT,11945.00
POSTAGE,10787.95
PARTY BUNTING,10227.60
Manual,8832.41
PAPER CHAIN KIT 50'S CHRISTMAS,7745.12
HEART OF WICKER SMALL,7641.50


## 4. Признаки для прогноза

Агрегируем по дням и добавляем календарные признаки и лаги. Лаги считаются только по прошлым дням — утечки будущего нет.

In [10]:
daily = cleaned.set_index("InvoiceDate")["Revenue"].resample("D").sum().to_frame()
daily = daily[daily["Revenue"] > 0].copy()

daily["dayofweek"] = daily.index.dayofweek
daily["month"] = daily.index.month
daily["is_weekend"] = (daily["dayofweek"] >= 5).astype(int)
daily["lag1"] = daily["Revenue"].shift(1)
daily["lag7"] = daily["Revenue"].shift(7)
daily["rolling7"] = daily["Revenue"].shift(1).rolling(7).mean()
daily = daily.dropna()

print("days:", len(daily), "period:", daily.index.min().date(), "-", daily.index.max().date())
daily.tail()

days: 597 period: 2009-12-08 - 2011-12-09


,Revenue,dayofweek,month,is_weekend,lag1,lag7,rolling7
InvoiceDate,,,,,,,
2011-12-05,5778.93,0,12,0,2014.01,1499.61,3207.058571
2011-12-06,4143.10,1,12,0,5778.93,4558.52,3818.390000
2011-12-07,7062.70,2,12,0,4143.10,4039.03,3759.044286
2011-12-08,3270.62,3,12,0,7062.70,3559.10,4190.997143
2011-12-09,1179.06,4,12,0,3270.62,4011.39,4149.785714


## 5. Модели и метрики

Baseline — «завтра будет как сегодня» (`lag1`). Модель — градиентный бустинг на календарных признаках и лагах. Сравниваем MAE и RMSE на последних 20% дней.

In [11]:
features = ["dayofweek", "month", "is_weekend", "lag1", "lag7", "rolling7"]
X = daily[features]
y = daily["Revenue"]

split = int(len(daily) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

model = GradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

baseline_pred = X_test["lag1"]
print("test days:", len(y_test))

test days: 120


In [12]:
def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    return {"mae": round(float(mae), 2), "rmse": round(float(rmse), 2), "mape": round(mape, 2)}

result_baseline = metrics(y_test, baseline_pred)
result_model = metrics(y_test, pred)
print("baseline:", result_baseline)
print("model:   ", result_model)

summary = {"baseline": result_baseline, "model": result_model, "test_days": int(len(y_test))}
(OUTPUTS / "metrics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

predictions = pd.DataFrame({"date": y_test.index, "actual": y_test.values, "predicted": pred})
predictions.to_csv(OUTPUTS / "predictions.csv", index=False)
joblib.dump(model, MODELS / "model.pkl")
print("saved metrics, predictions and model")

baseline: {'mae': 1571.69, 'rmse': 2035.76, 'mape': 59.37}
model:    {'mae': 1147.22, 'rmse': 1485.67, 'mape': 41.92}
saved metrics, predictions and model


In [13]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_test.index, y_test.values, label="факт", linewidth=1.5)
ax.plot(y_test.index, pred, label="прогноз", linewidth=1.5, alpha=0.85)
ax.set_title("Дневная выручка: факт и прогноз")
ax.set_ylabel("£")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "actual_vs_predicted.png", dpi=150)
plt.show()

C:\Users\boroz\AppData\Local\Temp\ipykernel_12416\1021421849.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 3.5))
importance.sort_values().plot.barh(ax=ax, color="#72B7B2")
ax.set_title("Важность признаков")
fig.tight_layout()
fig.savefig(FIGURES / "feature_importance.png", dpi=150)
plt.show()

importance.to_frame("importance")

C:\Users\boroz\AppData\Local\Temp\ipykernel_12416\1263793231.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,importance
rolling7,0.423620
lag1,0.227459
dayofweek,0.140705
lag7,0.119583
month,0.088145
is_weekend,0.000490


## 6. Выводы

- Данные грязные: отмены, возвраты, нулевые цены и пустые клиенты — считаем и удаляем явно, чтобы не тащить их в модель.
- Выручка сезонна: пик — ноябрь-декабрь, провал — январь; модель это ловит через месяц и лаги.
- Бустинг сравнивается с наивным baseline на тех же днях; метрики сохраняются в `outputs/metrics.json`.
- Ограничения: прогноз по одному магазину, без промо и праздников; горизонт — один день; для длинного горизонта нужны внешние признаки.

### Что дальше

- добавить праздники и промо-календарь;
- попробовать прогноз на неделю вперёд с рекурсивными лагами;
- считать выручку по категориям и строить отдельные модели.
